# UK Retail Market-Basket Intelligence

**Reproducible research walkthrough.** UCI Online Retail, a historical single UK-based non-store retailer (2010–2011), not representative UK supermarket data.

This notebook reproduces data cleaning, chronological basket-completion benchmarks, paired uncertainty and charts. Results are generated only when cells execute against the original workbook.

In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src' / 'data.py').exists()), None)
if ROOT is None:
    raise FileNotFoundError('Project source files are missing. Download and extract the entire GitHub repository, then open this notebook inside it: https://github.com/AlDSChriSOB/UK-Retail-Intelligence-Apriori')
sys.path.insert(0, str(ROOT))
import pandas as pd
import matplotlib.pyplot as plt
from src.data import load_transactions, clean_transactions, download_dataset, extract_dataset
from src.chronological_holdout import invoice_matrix
from src.recommendation_benchmark import benchmark
from src.paired_bootstrap import paired_bootstrap

WORKBOOK = ROOT / 'data/raw/Online Retail.xlsx'
if not WORKBOOK.exists():
    print('Downloading official UCI Online Retail dataset (first run only)...')
    WORKBOOK = extract_dataset(download_dataset(zip_path=ROOT / 'data/raw/online-retail.zip', extract_dir=ROOT / 'data/raw'))
OUT = ROOT / 'results/figures'
OUT.mkdir(parents=True, exist_ok=True)


## Data provenance and cleaning
Invoice-level product presence is binary. Cancellation invoices, nonpositive quantities and nonpositive prices are excluded; missing CustomerID is retained.

In [ ]:
transactions, cleaning = clean_transactions(load_transactions(WORKBOOK))
display(pd.Series(cleaning, name='count').to_frame())
invoice_metadata, product_codes, baskets = invoice_matrix(transactions)
print('Invoice matrix:', baskets.shape, 'nonzero entries:', baskets.nnz)

## Chronological masked-basket recommendation benchmark
The first 70% of invoices train all models. On later invoices with at least two items, one randomly chosen product is hidden and the remainder is the query. All models exclude observed items from recommendations. This is **basket completion**, not next-visit prediction.

In [ ]:
outcomes = ROOT / 'reports/generated/per_invoice_hits.csv'
report = benchmark(baskets, train_fraction=0.7, seed=42, outcome_csv=outcomes)
records = [{'method': method, 'k': int(k), **values} for method, metrics in report['metrics'].items() for k, values in metrics.items()]
metrics = pd.DataFrame(records)
display(metrics)

## Publication-ready figure (recomputed, not hard-coded)
Hit Rate@K is the proportion of eligible test invoices for which the single hidden product appears among top-K suggestions.

In [ ]:
plot = metrics.pivot(index='k', columns='method', values='hit_rate_at_k') * 100
ax = plot.plot(kind='bar', figsize=(8, 5), rot=0)
ax.set(xlabel='Recommendation cutoff K', ylabel='Hit rate (%)', title='Chronological held-out basket completion')
ax.legend(title='Training-only model'); ax.grid(axis='y', alpha=.25)
plt.tight_layout(); plt.savefig(OUT / 'hit_rate_by_method.png', dpi=220, bbox_inches='tight')
plt.show()

## Paired uncertainty on the same test invoices
Bootstrap resamples entire invoices to preserve method pairing. The 95% percentile interval is descriptive for this split and masking seed.

In [ ]:
hits = pd.read_csv(outcomes)
comparison = paired_bootstrap(hits['apriori_at_10'], hits['cooccurrence_at_10'], replicates=10000, seed=20261111)
display(pd.Series(comparison).to_frame('value'))
print('Interval crosses zero:', comparison['ci95_percentile'][0] <= 0 <= comparison['ci95_percentile'][1])

## Limitations and reproducibility
This is a single historical retailer and one chronological split; no customer-disjoint evaluation. One hidden item means Recall@K equals Hit Rate@K. Product variants and duplicate descriptions may affect rankings. The pair-rule method is **not** general multi-item Apriori. See the reports directory for seed, temporal and catalogue sensitivity findings.